## The goal is to find revenue growth opportunities
## Based on EDA

In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect('../data/database/data_mart.db') 

## Problem 1: Low customer retention

**Metric:**
- Avg orders per customer = 1,03, majority are one-time buyers

---

## Hypothesis 1: Categories impact retention

In [3]:
orders_per_unique_client = pd.read_sql(
    """
    SELECT
        product_category_name,
        COUNT(DISTINCT order_id) / CAST(COUNT(DISTINCT customer_unique_id) AS float) AS orders_per_unique_client        
    FROM data_mart
    GROUP BY product_category_name
    ORDER BY orders_per_unique_client DESC
    """, conn)
orders_per_unique_client.set_index('product_category_name', inplace=True)
orders_per_unique_client

,orders_per_unique_client
product_category_name,
artes_e_artesanato,1.095238
eletrodomesticos,1.086771
moveis_quarto,1.043956
casa_conforto_2,1.043478
fraldas_higiene,1.040000
...,...
cine_foto,1.000000
cds_dvds_musicais,1.000000
artigos_de_natal,1.000000


**Conclusion:** Not confirmed

---

## Hypothesis 2: Delivery time impacts retention

In [4]:
pd.read_sql(
    """
    SELECT AVG(delivery_days) AS median_delivery_days
    FROM (
        SELECT julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp) AS delivery_days
        FROM data_mart
        ORDER BY delivery_days
        LIMIT 2 - (SELECT COUNT(*) FROM data_mart) % 2
        OFFSET (SELECT (COUNT(*) - 1) / 2 FROM data_mart)) AS table1
    """,conn)


,median_delivery_days
0,10.048715


In [5]:
deliverytime_per_states = pd.read_sql(
    """
    SELECT 
        customer_state,
        AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)) AS delivery_days
    FROM data_mart
    GROUP BY customer_state
    ORDER BY delivery_days
    """,conn)
deliverytime_per_states.set_index('customer_state', inplace=True)
deliverytime_per_states

,delivery_days
customer_state,
SP,8.698845
PR,11.930657
MG,11.959896
DF,12.922145
SC,14.975430
RJ,15.132892
RS,15.178148
GO,15.407242
MS,15.534072


In [6]:
average_bill_per_state = pd.read_sql(
    """
    SELECT
        customer_state,
        AVG(revenue) AS average_bill
    FROM data_mart
    GROUP BY customer_state
    ORDER BY average_bill
    """, conn)
average_bill_per_state.set_index('customer_state', inplace=True)
average_bill_per_state

,average_bill
customer_state,
SP,124.780902
PR,139.810951
MG,141.467850
RS,141.909130
ES,143.754338
RJ,145.781767
SC,145.977744
DF,146.801222
GO,149.054735


**Result:**
- Median delivery time = 10 days  
- Regions with faster delivery - higher activity  
- Regions with 15+ days - lower activity  

**Conclusion:** Partially confirmed

---

## Hypothesis 3 – 4: сustomer experience and marketing impact retention

**Result:**
- No data on customer experience or marketing  

**Conclusion:** Not testable

---

## Final insight

- Retention is not driven by product categories  
- Likely influenced by delivery time  
- Requires further validation

In [7]:
conn.close()